# web scraping
with API method

In [12]:
import os
import time
import logging
import requests
import pandas as pd
from tqdm import tqdm

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────

API_KEY = "913252ae6b949507f85ecb4c380f698c"
TARGET_MOVIES = 6000   # 🔥 هنا زودنا العدد
OUTPUT_CSV = "movies_dataset.csv"
DELAY_SECONDS = 0.2
MAX_CAST = 5

BASE_URL = "https://api.themoviedb.org/3"

# ─────────────────────────────────────────────
# LOGGING
# ─────────────────────────────────────────────

logging.basicConfig(level=logging.INFO)
log = logging.getLogger(__name__)

# ─────────────────────────────────────────────
# API REQUEST
# ─────────────────────────────────────────────

def api_get(endpoint, params=None):
    url = BASE_URL + endpoint
    base_params = {"api_key": API_KEY, "language": "en-US"}

    if params:
        base_params.update(params)

    try:
        r = requests.get(url, params=base_params, timeout=10)
        r.raise_for_status()
        return r.json()
    except Exception as e:
        log.warning(f"Error: {e}")
        return None

# ─────────────────────────────────────────────
# GET MOVIE IDS (UPDATED 🔥)
# ─────────────────────────────────────────────

def fetch_movie_ids():
    movie_ids = set()

    sort_orders = [
        "popularity.desc",
        "vote_count.desc",
        "revenue.desc",
        "vote_average.desc"
    ]

    pbar = tqdm(total=TARGET_MOVIES, desc="Collecting IDs")

    for sort_by in sort_orders:
        page = 1

        while len(movie_ids) < TARGET_MOVIES:
            data = api_get("/discover/movie", {
                "sort_by": sort_by,
                "page": page,
                "vote_count.gte": 100
            })

            if not data or not data.get("results"):
                break

            for m in data["results"]:
                if m["id"] not in movie_ids:
                    movie_ids.add(m["id"])
                    pbar.update(1)

            page += 1
            time.sleep(DELAY_SECONDS)

            if page > 500:
                break

        if len(movie_ids) >= TARGET_MOVIES:
            break

    pbar.close()
    return list(movie_ids)[:TARGET_MOVIES]

# ─────────────────────────────────────────────
# GET DETAILS
# ─────────────────────────────────────────────

def fetch_movie_details(movie_id):
    data = api_get(f"/movie/{movie_id}", {
        "append_to_response": "credits"
    })

    if not data:
        return None

    genres = ", ".join([g["name"] for g in data.get("genres", [])])

    cast = [
        c["name"]
        for c in sorted(data["credits"]["cast"], key=lambda x: x["order"])
    ][:MAX_CAST]

    director = [
        c["name"]
        for c in data["credits"]["crew"]
        if c["job"] == "Director"
    ]

    return {
        "id": movie_id,
        "title": data.get("title"),
        "rating": data.get("vote_average"),
        "year": (data.get("release_date") or "")[:4],
        "genres": genres,
        "overview": data.get("overview"),
        "cast": " | ".join(cast),
        "director": " | ".join(director),
    }

# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────

def main():
    log.info("🚀 Start collecting movies...")

    ids = fetch_movie_ids()
    log.info(f"Collected {len(ids)} IDs")

    records = []

    for mid in tqdm(ids, desc="Fetching details"):
        d = fetch_movie_details(mid)
        if d:
            records.append(d)
        time.sleep(DELAY_SECONDS)

    df = pd.DataFrame(records)

    # تنظيف
    df = df.dropna(subset=["title", "overview"])
    df = df.drop_duplicates()

    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

    log.info(f"✅ Saved {len(df)} movies to CSV")

# ─────────────────────────────────────────────

if __name__ == "__main__":
    main()

08:21:48  INFO      🚀 Start collecting movies...
08:26:06  INFO      Collected 6000 IDs
Fetching details: 100%|██████████| 6000/6000 [1:24:49<00:00,  1.18it/s]  
09:50:55  INFO      ✅ Saved 5997 movies to CSV
